In [0]:
# Cell: Install Packages
%pip install torch torchvision matplotlib opencv-python lmdb pandas scipy loguru tikzplotlib jpeg4py

## Load the SAMURAI SAM 2 MODEL

In [0]:
%sh
rm -rf /tmp/samurai
git clone https://github.com/yangchris11/samurai.git /tmp/samurai

Cloning into '/tmp/samurai'...


In [0]:
%sh
cd /tmp/samurai/sam2
pip install -e .
pip install -e ".[notebooks]"

In [0]:
dbutils.library.restartPython()

In [0]:
try:
    from sam2.build_sam import build_sam2_video_predictor
    print("Successfully imported build_sam_video_predictor from sam2.build_sam!")
except Exception as e:
    print("Error importing SAM2 module:", e)

Successfully imported build_sam_video_predictor from sam2.build_sam!


Contour Annotations for multiple objects

In [0]:
import argparse
import os
import os.path as osp
import numpy as np
import cv2
import torch
import gc
import sys
import json
import time
import shutil

from sam2.build_sam import build_sam2_video_predictor

def load_txt(gt_path):
    """
    Loads bounding box prompts from a text file formatted as:
       <object_name>: x,y,w,h
    If the file is not found at the given path, it tries prepending '/dbfs/'.
    Returns a dictionary mapping object names to a tuple: ((x1,y1,x2,y2), 0).
    """
    if not osp.exists(gt_path):
        gt_path_dbfs = os.path.join("/dbfs", gt_path.lstrip("/"))
        if osp.exists(gt_path_dbfs):
            gt_path = gt_path_dbfs
        else:
            raise FileNotFoundError(f"File not found: {gt_path}")
    with open(gt_path, 'r') as f:
        lines = f.readlines()
    prompts = {}
    for line in lines:
        parts = line.strip().split(":")
        if len(parts) != 2:
            raise ValueError("Each line must be formatted as <object_name>: x,y,w,h")
        obj_name = parts[0].strip()
        coords = parts[1].strip().split(",")
        if len(coords) != 4:
            raise ValueError("Coordinates must have four comma-separated values")
        x, y, w, h = map(float, coords)
        # Convert to (x1,y1,x2,y2)
        prompts[obj_name] = ((int(x), int(y), int(x+w), int(y+h)), 0)
    return prompts

def determine_model_cfg(model_path):
    """
    Returns the configuration file for the model.
    For example, if "large" is in the model_path, returns:
      "configs/sam2.1/sam2.1_hiera_l.yaml"
    """
    if "large" in model_path:
        return "configs/sam2.1/sam2.1_hiera_l.yaml"
    elif "base_plus" in model_path:
        return "configs/sam2.1/sam2.1_hiera_b+.yaml"
    elif "small" in model_path:
        return "configs/sam2.1/sam2.1_hiera_s.yaml"
    elif "tiny" in model_path:
        return "configs/sam2.1/sam2.1_hiera_t.yaml"
    else:
        raise ValueError("Unknown model size in path!")

def prepare_frames_or_path(video_path):
    """
    Validates that video_path is either a .mp4 file or a directory of JPEG images.
    If the provided path does not exist, tries to prepend '/dbfs/'.
    """
    if osp.exists(video_path):
        return video_path
    dbfs_path = os.path.join("/dbfs", video_path.lstrip("/"))
    if osp.exists(dbfs_path):
        return dbfs_path
    raise ValueError("Invalid video_path format. Should be a .mp4 file or a directory of jpg frames.")

def copy_to_filestore(src, dest, max_retries=3, delay=5):
    """
    Copies a file from src to dest using a shell command.
    Both src and dest should have the /dbfs prefix.
    Retries up to max_retries if the file is not found at the destination.
    """
    for attempt in range(max_retries):
        cmd = f"cp '{src}' '{dest}'"
        print(f"Copy attempt {attempt+1}: {cmd}")
        os.system(cmd)
        time.sleep(delay)
        if osp.exists(dest):
            print(f"File successfully copied to: {dest}")
            return True
        else:
            print(f"Attempt {attempt+1} failed.")
    return False

def process_folder(frames_dir, predictor, current_prompt, object_names):
    """
    Processes a single folder of frames.
    
    Arguments:
      frames_dir: directory of JPEG frames for this batch.
      predictor: the SAM2 predictor (pre-built).
      current_prompt: dict mapping obj_id to prompt bounding box ([x1,y1,x2,y2]) for frame 0.
      object_names: list of object names corresponding to obj_id.
      
    Returns:
      batch_annotations: dict { obj_id: { frame_name: annotation, ... } }
      new_prompt: dict mapping obj_id to new prompt (bounding box from the last frame) in [x1,y1,x2,y2] format.
    """
    # List and sort frame files in this folder.
    frame_files = sorted([osp.join(frames_dir, f) for f in os.listdir(frames_dir)
                          if f.lower().endswith((".jpg", ".jpeg"))])
    num_frames = len(frame_files)
    print(f"Processing {num_frames} frames in folder: {frames_dir}")
    
    # Initialize state for this folder.
    state = predictor.init_state(frames_dir, async_loading_frames=True, offload_video_to_cpu=True)
    
    # For each object, add the initial prompt at frame 0.
    for obj_id, bbox in current_prompt.items():
        predictor.add_new_points_or_box(state, box=bbox, frame_idx=0, obj_id=obj_id)
    
    batch_annotations = {obj_id: {} for obj_id in current_prompt.keys()}
    
    # Process each frame in this folder.
    for local_idx, object_ids, masks in predictor.propagate_in_video(state):
        frame_path = frame_files[local_idx]
        frame_name = osp.basename(frame_path)  # e.g., "003501.jpg"
        global_frame = frame_name  # using filename as global identifier
        
        for obj_id, mask in zip(object_ids, masks):
            mask_np = mask[0].cpu().numpy()
            mask_binary = mask_np > 0.0
            nonzero_pixels = np.argwhere(mask_binary)
            if len(nonzero_pixels) == 0:
                bbox_coords = [0, 0, 0, 0]
            else:
                y_min, x_min = nonzero_pixels.min(axis=0).tolist()
                y_max, x_max = nonzero_pixels.max(axis=0).tolist()
                bbox_coords = [x_min, y_min, x_max - x_min, y_max - y_min]  # [x, y, w, h]
            # Extract contour for precise annotation.
            contours, _ = cv2.findContours((mask_binary*255).astype(np.uint8),
                                           cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contour_list = [c.squeeze().tolist() for c in contours if len(c) > 2]
            batch_annotations[obj_id][global_frame] = {
                "bounding_box": bbox_coords,
                "contour": contour_list,
                "object_name": object_names[obj_id]
            }
    
    # Determine new prompt from the last frame of this folder.
    new_prompt = {}
    for obj_id in current_prompt.keys():
        last_frame = sorted(batch_annotations[obj_id].keys())[-1]
        ann = batch_annotations[obj_id][last_frame]
        bb = ann["bounding_box"]  # [x, y, w, h]
        new_prompt[obj_id] = (bb[0], bb[1], bb[0] + bb[2], bb[1] + bb[3])
    
    del state
    gc.collect()
    torch.cuda.empty_cache()
    return batch_annotations, new_prompt

def main(args):
    # Determine the configuration file.
    model_cfg = determine_model_cfg(args.model_path)
    
    # Build the predictor (to be reused across folders).
    predictor = build_sam2_video_predictor(model_cfg, args.model_path, device="cuda:0")
    
    # Get the list of frame folders from a comma-separated string.
    folders = [folder.strip() for folder in args.frames_dirs.split(",")]
    folders = [prepare_frames_or_path(folder) for folder in folders]
    
    # Load initial bounding box prompts.
    prompts = load_txt(args.txt_path)
    object_names = sorted(prompts.keys())
    obj_name_to_id = {name: idx for idx, name in enumerate(object_names)}
    
    # For the first folder, current_prompt comes from the file.
    current_prompt = {}
    for obj_name, prompt in prompts.items():
        current_prompt[obj_name_to_id[obj_name]] = prompt[0]
    
    # Process each folder sequentially.
    for folder in folders:
        folder_basename = osp.basename(osp.normpath(folder))
        output_dir = osp.join(args.output_base_dir, folder_basename)
        os.makedirs(output_dir, exist_ok=True)
        print(f"Processing folder: {folder_basename} (Output dir: {output_dir})")
        
        batch_ann, new_prompt = process_folder(folder, predictor, current_prompt, object_names)
        # Check if we got any annotations
        for obj_id, ann in batch_ann.items():
            print(f"Object '{object_names[obj_id]}' has {len(ann)} frames annotated in folder '{folder_basename}'.")
            # Save JSON for this object in this folder.
            ann_filename = f"annotations_{object_names[obj_id]}.json"
            ann_full_path = osp.join(output_dir, ann_filename)
            with open(ann_full_path, "w") as f:
                json.dump(ann, f)
            print(f"Saved annotations for object '{object_names[obj_id]}' to {ann_full_path}")
        
        # Update current_prompt for next folder.
        current_prompt = new_prompt
        
        # Clear GPU memory between folders.
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Completed processing folder {folder_basename}.\n")
    
    del predictor
    gc.collect()
    torch.clear_autocast_cache()
    torch.cuda.empty_cache()
    print("Batch processing complete. All annotations saved.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    # Comma-separated list of frame directories. Paths to frams directory one by one in the order you want to track
    parser.add_argument("--frames_dirs", default= "xxx/video1," 
                                                  "xxx/video2,"
                                                  "xxx/video3,"
                                                  "xxx/video4,"
                                                  "xxx/video5,"
                                                  "xxx/video6,"
                                                  "xxx/....,"
                                                  "xxx/video100,",
                        help="Comma-separated list of frame directories.")
    parser.add_argument("--txt_path", default="/xxx/BoundingBoxes_Eem_0619_0603_30.txt", # path to the saved bounding boxes from Object Detection
                        help="Path to initial bounding box text file formatted as <object_name>: x,y,w,h")
    parser.add_argument("--model_path", default="/dbfs/mnt/playbehavior/SAM2 weights/sam2.1_hiera_large.pt",
                        help="Path to the model checkpoint.")
    # Output base directory for annotations.
    parser.add_argument("--output_base_dir", default="/xxx/Eem_ch04_0619_060343_235956", # Output path to save the mask annotations
                        help="Base directory to save annotation JSON files (one subfolder per frames directory).")
    args, unknown = parser.parse_known_args()
    main(args)